In [136]:
import numpy as np
from sklearn.metrics import cohen_kappa_score
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import pandas as pd

discussion_paths = "data/discussion_full_data.xlsx"
repo_card_path = "data/cards_full_data.xlsx"
commit_path = "data/commits_full_data.xlsx"

sheet = "AIA"  
sheet2 = "ACT"  
sheet3 = "RG"  


discussion_AIA = pd.read_excel(discussion_paths, sheet_name=sheet)
repo_card_AIA = pd.read_excel(repo_card_path, sheet_name=sheet)
commits_AIA = pd.read_excel(commit_path, sheet_name=sheet)

discussion_ACT = pd.read_excel(discussion_paths, sheet_name=sheet2)
repo_card_ACT = pd.read_excel(repo_card_path, sheet_name=sheet2)
commits_ACT = pd.read_excel(commit_path, sheet_name=sheet2)

repo_card_RG = pd.read_excel(repo_card_path, sheet_name=sheet3)

print(f"commits shape {commits_AIA.shape}")
print(f"discussion shape {discussion_AIA.shape}") 
print(f"repo card shape {repo_card_AIA.shape}")

print(f"commits shape ACT {commits_ACT.shape}")
print(f"discussion shape ACT {discussion_ACT.shape}") 
print(f"repo card shape ACT {repo_card_ACT.shape}")

print(f"repo card shape RG {repo_card_RG.shape}")


# Useful_repo_id looks like "<type>/<owner>/<name>"
def split_repo_id(df):
    parts = df["Useful_repo_id"].str.split("/", n=2, expand=True)
    df["repo_type"] = parts[0]
    df["owner"] = parts[1]
    df["name"] = parts[2]
    return df

# ---------------------------------------------------------- Discussions
discussion_AIA = split_repo_id(discussion_AIA)
discussion_ACT = split_repo_id(discussion_ACT)

# # ---------------------------------------------------------- Repo cards
repo_card_AIA = split_repo_id(repo_card_AIA)
repo_card_ACT = split_repo_id(repo_card_ACT)
repo_card_RG = split_repo_id(repo_card_RG)

# ---------------------------------------------------------- Commits
commits_AIA = split_repo_id(commits_AIA)
commits_ACT = split_repo_id(commits_ACT)

commits shape (1093, 8)
discussion shape (17, 6)
repo card shape (291, 6)
commits shape ACT (166, 9)
discussion shape ACT (56, 6)
repo card shape ACT (1705, 10)
repo card shape RG (4, 7)


In [137]:
# --------------- Discussion
discussion_AIA = discussion_AIA[['Useful_repo_id','discussion_url','CD2', 'CD1', 'Final Labels','Language','repo_type', 'owner', 'name' ]].copy()
discussion_ACT = discussion_ACT[['Useful_repo_id','discussion_url','CD2', 'CD1', 'Final Labels','Language','repo_type', 'owner', 'name' ]].copy()
# --------------- Commits
commits_ACT = commits_ACT[['Useful_repo_id','commit_url','CD2', 'CD1', 'Final Labels','Language','repo_type', 'owner', 'name' ]].copy()
commits_AIA = commits_AIA[['Useful_repo_id','commit_url','CD2', 'CD1', 'Final Labels','Language','repo_type', 'owner', 'name' ]].copy()
# --------------- Repo card
repo_card_ACT = repo_card_ACT[['Useful_repo_id','card_url','CD2', 'CD1', 'Final Labels','Language','repo_type', 'owner', 'name' ]].copy()
repo_card_AIA = repo_card_AIA[['Useful_repo_id','card_url','CD2', 'CD1', 'Final Labels','Language','repo_type', 'owner', 'name' ]].copy()
repo_card_RG = repo_card_RG[['Useful_repo_id','card_url','CD2', 'CD1', 'Final Labels','Language','repo_type', 'owner', 'name' ]].copy()

In [138]:
# --------------- Discussions
discussions = pd.concat(
    [discussion_AIA.assign(pattern="AIA"),
     discussion_ACT.assign(pattern="AI Act")],
    ignore_index=True,
)

# --------------- Commits
commits = pd.concat(
    [commits_AIA.assign(pattern="AIA"),
     commits_ACT.assign(pattern="AI Act")],
    ignore_index=True,
)

# --------------- Repo cards
cards = pd.concat(
    [repo_card_AIA.assign(pattern="AIA"),
     repo_card_ACT.assign(pattern="AI Act"),
     repo_card_RG.assign(pattern="2024 1689")],
    ignore_index=True,
)

In [139]:
discussions.shape, commits.shape, cards.shape

((73, 10), (1259, 10), (2000, 10))

In [140]:
before = {"discussions": len(discussions), "commits": len(commits), "cards": len(cards)}

discussions = discussions.drop_duplicates(subset="discussion_url", keep="first").reset_index(drop=True)
commits = commits.drop_duplicates(subset="commit_url", keep="first").reset_index(drop=True)
cards = cards.drop_duplicates(subset="card_url", keep="first").reset_index(drop=True)

after = {"discussions": len(discussions), "commits": len(commits), "cards": len(cards)}
for name in before:
    print(f"{name}: {before[name]} → {after[name]} ({before[name] - after[name]} duplicates removed)")

discussions: 73 → 73 (0 duplicates removed)
commits: 1259 → 1259 (0 duplicates removed)
cards: 2000 → 2000 (0 duplicates removed)


In [141]:
label_map = {
    "regulated":           "Compliance and responsibility",
    "regulated*":            "Non applicability claims",
    "false positives":       "false positives",
    "artifact":             "Compliance artifacts",
    "understanding":        "Knowledge resources",
    "regulated/violation": "Reported alleged violations",
}

In [142]:
cards["Final Labels"].value_counts()

Final Labels
Regulated              1129
False positives         605
Understanding           135
ARTIFACT                119
Regulated*                9
Regulated/violation       2
false positives           1
Name: count, dtype: int64

In [143]:
# commits ----------------------------------------
normalized = commits["Final Labels"].astype(str).str.strip().str.lower()
normalized_CD1 = commits["CD1"].astype(str).str.strip().str.lower()
normalized_CD2 = commits["CD2"].astype(str).str.strip().str.lower()

commits["Label_tag"] = commits["Final Labels"]
commits["CD1_tag"] = commits["CD1"]
commits["CD2_tag"] = commits["CD2"]

commits["Final Labels"] = normalized.map(label_map)
commits["CD1"] = normalized_CD1.map(label_map)
commits["CD2"] = normalized_CD2.map(label_map)

unmatched_commits = commits.loc[commits["Final Labels"].isna() & commits["Label_tag"].notna(), "Label_tag"].unique()
unmatched_commits_CD1 = commits.loc[commits["CD1"].isna() & commits["CD1_tag"].notna(), "Label_tag"].unique()
unmatched_commits_CD2 = commits.loc[commits["CD2"].isna() & commits["CD2_tag"].notna(), "Label_tag"].unique()


# discussions ----------------------------------------
normalized = discussions["Final Labels"].astype(str).str.strip().str.lower()
normalized_CD1 = discussions["CD1"].astype(str).str.strip().str.lower()
normalized_CD2 = discussions["CD2"].astype(str).str.strip().str.lower()

discussions["Label_tag"] = discussions["Final Labels"]
discussions["CD1_tag"] = discussions["CD1"]
discussions["CD2_tag"] = discussions["CD2"]

discussions["Final Labels"] = normalized.map(label_map)
discussions["CD1"] = normalized_CD1.map(label_map)
discussions["CD2"] = normalized_CD2.map(label_map)

unmatched_discussions = discussions.loc[discussions["Final Labels"].isna() & discussions["Label_tag"].notna(), "Label_tag"].unique()
unmatched_discussions_CD1 = discussions.loc[discussions["CD1"].isna() & discussions["CD1_tag"].notna(), "Label_tag"].unique()
unmatched_discussions_CD2 = discussions.loc[discussions["CD2"].isna() & discussions["CD2_tag"].notna(), "Label_tag"].unique()


# cards ----------------------------------------
normalized = cards["Final Labels"].astype(str).str.strip().str.lower()
normalized_CD1 = cards["CD1"].astype(str).str.strip().str.lower()
normalized_CD2 = cards["CD2"].astype(str).str.strip().str.lower()

cards["Label_tag"] = cards["Final Labels"]
cards["CD1_tag"] = normalized_CD1.map(label_map)
cards["CD2_tag"] = normalized_CD2.map(label_map)

cards["Final Labels"] = normalized.map(label_map)
cards["CD1"] = normalized_CD1.map(label_map)
cards["CD2"] = normalized_CD2.map(label_map)

unmatched_cards = cards.loc[cards["Final Labels"].isna() & cards["Label_tag"].notna(), "Label_tag"].unique()
unmatched_cards_CD1 = cards.loc[cards["Final Labels"].isna() & cards["CD1_tag"].notna(), "Label_tag"].unique()
unmatched_cards_CD2 = cards.loc[cards["Final Labels"].isna() & cards["CD2_tag"].notna(), "Label_tag"].unique()


In [144]:
cards['CD2'].value_counts()

CD2
Compliance and responsibility    1137
false positives                   594
Knowledge resources               146
Compliance artifacts              109
Non applicability claims            9
Reported alleged violations         5
Name: count, dtype: int64

In [94]:
cards["Final Labels"].value_counts()

Final Labels
Regulated              1129
False positives         605
Understanding           135
ARTIFACT                119
Regulated*                9
Regulated/violation       2
false positives           1
Name: count, dtype: int64

In [72]:
commits['Final Labels'].value_counts()

Final Labels
false positives                  1127
Compliance and responsibility      63
Compliance artifacts               35
Knowledge resources                34
Name: count, dtype: int64

In [104]:
discussions['Final Labels'].value_counts()

Final Labels
false positives                  32
Compliance and responsibility    16
Reported alleged violations      15
Knowledge resources               7
Compliance artifacts              3
Name: count, dtype: int64

In [105]:
cards['Final Labels'].value_counts()

Final Labels
Compliance and responsibility    1129
false positives                   606
Knowledge resources               135
Compliance artifacts              119
Non applicability claims            9
Reported alleged violations         2
Name: count, dtype: int64

In [146]:
CATEGORIES = [
    "Compliance and responsibility",
    "Non applicability claims",
    "false positives",
    "Compliance artifacts",
    "Knowledge resources",
    "Reported alleged violations",
]

# 1. Clean labels so trivial differences don't count as disagreements
for col in ["CD1", "CD2"]:
    cards[col] = cards[col].astype(str).str.strip()

unknown = set(cards["CD1"]).union(cards["CD2"]) - set(CATEGORIES)
if unknown:
    print("Warning: labels not in the category list:", unknown)

c1, c2 = cards["CD1"], cards["CD2"]

# 2. Overall agreement
pct_agreement = (c1 == c2).mean() * 100
kappa = cohen_kappa_score(c1, c2, labels=CATEGORIES)
print(f"Records: {len(cards)}")
print(f"Percent agreement: {pct_agreement:.2f}%")
print(f"Cohen's kappa: {kappa:.3f}")

# 3. 95% confidence interval for kappa (bootstrap)
rng = np.random.default_rng(42)
boot = []
for _ in range(2000):
    idx = rng.integers(0, len(cards), len(cards))
    boot.append(cohen_kappa_score(c1.iloc[idx], c2.iloc[idx], labels=CATEGORIES))
low, high = np.nanpercentile(boot, [2.5, 97.5])
print(f"95% CI: [{low:.3f}, {high:.3f}]")

# 4. Confusion matrix: rows = coder 1, columns = coder 2
confusion = pd.crosstab(c1, c2, rownames=["CD1"], colnames=["CD2"], dropna=False)
confusion = confusion.reindex(index=CATEGORIES, columns=CATEGORIES, fill_value=0)
print(confusion)

# 5. Per-category agreement (positive specific agreement)
rows = []
for cat in CATEGORIES:
    both = ((c1 == cat) & (c2 == cat)).sum()
    either = (c1 == cat).sum() + (c2 == cat).sum()
    rows.append({
        "category": cat,
        "coder1_n": (c1 == cat).sum(),
        "coder2_n": (c2 == cat).sum(),
        "agreement_%": round(2 * both / either * 100, 2) if either else np.nan,
    })
print(pd.DataFrame(rows))

Records: 2000
Percent agreement: 95.15%
Cohen's kappa: 0.916
95% CI: [0.900, 0.932]
CD2                            Compliance and responsibility  \
CD1                                                            
Compliance and responsibility                           1112   
Non applicability claims                                   0   
false positives                                            9   
Compliance artifacts                                      10   
Knowledge resources                                        6   
Reported alleged violations                                0   

CD2                            Non applicability claims  false positives  \
CD1                                                                        
Compliance and responsibility                         0                3   
Non applicability claims                              8                0   
false positives                                       1              582   
Compliance artifacts   

In [147]:
CATEGORIES = [
    "Compliance and responsibility",
    "Non applicability claims",
    "false positives",
    "Compliance artifacts",
    "Knowledge resources",
    "Reported alleged violations",
]

# 1. Clean labels so trivial differences don't count as disagreements
for col in ["CD1", "CD2"]:
    commits[col] = commits[col].astype(str).str.strip()

unknown = set(commits["CD1"]).union(commits["CD2"]) - set(CATEGORIES)
if unknown:
    print("Warning: labels not in the category list:", unknown)

c1, c2 = commits["CD1"], commits["CD2"]

# 2. Overall agreement
pct_agreement = (c1 == c2).mean() * 100
kappa = cohen_kappa_score(c1, c2, labels=CATEGORIES)
print(f"Records: {len(commits)}")
print(f"Percent agreement: {pct_agreement:.2f}%")
print(f"Cohen's kappa: {kappa:.3f}")

# 3. 95% confidence interval for kappa (bootstrap)
rng = np.random.default_rng(42)
boot = []
for _ in range(2000):
    idx = rng.integers(0, len(commits), len(commits))
    boot.append(cohen_kappa_score(c1.iloc[idx], c2.iloc[idx], labels=CATEGORIES))
low, high = np.nanpercentile(boot, [2.5, 97.5])
print(f"95% CI: [{low:.3f}, {high:.3f}]")

# 4. Confusion matrix: rows = coder 1, columns = coder 2
confusion = pd.crosstab(c1, c2, rownames=["CD1"], colnames=["CD2"], dropna=False)
confusion = confusion.reindex(index=CATEGORIES, columns=CATEGORIES, fill_value=0)
print(confusion)

# 5. Per-category agreement (positive specific agreement)
rows = []
for cat in CATEGORIES:
    both = ((c1 == cat) & (c2 == cat)).sum()
    either = (c1 == cat).sum() + (c2 == cat).sum()
    rows.append({
        "category": cat,
        "coder1_n": (c1 == cat).sum(),
        "coder2_n": (c2 == cat).sum(),
        "agreement_%": round(2 * both / either * 100, 2) if either else np.nan,
    })
print(pd.DataFrame(rows))

Records: 1259
Percent agreement: 98.01%
Cohen's kappa: 0.898
95% CI: [0.859, 0.934]
CD2                            Compliance and responsibility  \
CD1                                                            
Compliance and responsibility                             56   
Non applicability claims                                   0   
false positives                                            4   
Compliance artifacts                                       3   
Knowledge resources                                        8   
Reported alleged violations                                0   

CD2                            Non applicability claims  false positives  \
CD1                                                                        
Compliance and responsibility                         0                0   
Non applicability claims                              0                0   
false positives                                       0             1123   
Compliance artifacts   

In [148]:
CATEGORIES = [
    "Compliance and responsibility",
    "Non applicability claims",
    "false positives",
    "Compliance artifacts",
    "Knowledge resources",
    "Reported alleged violations",
]

# 1. Clean labels so trivial differences don't count as disagreements
for col in ["CD1", "CD2"]:
    discussions[col] = discussions[col].astype(str).str.strip()

unknown = set(discussions["CD1"]).union(discussions["CD2"]) - set(CATEGORIES)
if unknown:
    print("Warning: labels not in the category list:", unknown)

c1, c2 = discussions["CD1"], discussions["CD2"]

# 2. Overall agreement
pct_agreement = (c1 == c2).mean() * 100
kappa = cohen_kappa_score(c1, c2, labels=CATEGORIES)
print(f"Records: {len(discussions)}")
print(f"Percent agreement: {pct_agreement:.2f}%")
print(f"Cohen's kappa: {kappa:.3f}")

# 3. 95% confidence interval for kappa (bootstrap)
rng = np.random.default_rng(42)
boot = []
for _ in range(2000):
    idx = rng.integers(0, len(discussions), len(discussions))
    boot.append(cohen_kappa_score(c1.iloc[idx], c2.iloc[idx], labels=CATEGORIES))
low, high = np.nanpercentile(boot, [2.5, 97.5])
print(f"95% CI: [{low:.3f}, {high:.3f}]")

# 4. Confusion matrix: rows = coder 1, columns = coder 2
confusion = pd.crosstab(c1, c2, rownames=["CD1"], colnames=["CD2"], dropna=False)
confusion = confusion.reindex(index=CATEGORIES, columns=CATEGORIES, fill_value=0)
print(confusion)

# 5. Per-category agreement (positive specific agreement)
rows = []
for cat in CATEGORIES:
    both = ((c1 == cat) & (c2 == cat)).sum()
    either = (c1 == cat).sum() + (c2 == cat).sum()
    rows.append({
        "category": cat,
        "coder1_n": (c1 == cat).sum(),
        "coder2_n": (c2 == cat).sum(),
        "agreement_%": round(2 * both / either * 100, 2) if either else np.nan,
    })
print(pd.DataFrame(rows))

Records: 73
Percent agreement: 84.93%
Cohen's kappa: 0.786
95% CI: [0.663, 0.887]
CD2                            Compliance and responsibility  \
CD1                                                            
Compliance and responsibility                             15   
Non applicability claims                                   0   
false positives                                            0   
Compliance artifacts                                       0   
Knowledge resources                                        3   
Reported alleged violations                                0   

CD2                            Non applicability claims  false positives  \
CD1                                                                        
Compliance and responsibility                         0                0   
Non applicability claims                              0                0   
false positives                                       0               32   
Compliance artifacts     

In [149]:
all_records = pd.concat(
    [cards.rename(columns={"card_url": "url"}).assign(source="Cards"),
     discussions.rename(columns={"discussion_url": "url"}).assign(source="Discussions"),
     commits.rename(columns={"commit_url": "url"}).assign(source="Commits")],
    ignore_index=True,
)

# 1. Clean labels so trivial differences don't count as disagreements
for col in ["CD1", "CD2"]:
    all_records[col] = all_records[col].astype(str).str.strip()

unknown = set(all_records["CD1"]).union(all_records["CD2"]) - set(CATEGORIES)
if unknown:
    print("Warning: labels not in the category list:", unknown)

c1, c2 = all_records["CD1"], all_records["CD2"]

# 2. Overall agreement
pct_agreement = (c1 == c2).mean() * 100
kappa = cohen_kappa_score(c1, c2, labels=CATEGORIES)
print(f"Records: {len(all_records)}")
print(f"Percent agreement: {pct_agreement:.2f}%")
print(f"Cohen's kappa: {kappa:.3f}")

# 3. 95% confidence interval for kappa (bootstrap)
rng = np.random.default_rng(42)
boot = []
for _ in range(2000):
    idx = rng.integers(0, len(all_records), len(all_records))
    boot.append(cohen_kappa_score(c1.iloc[idx], c2.iloc[idx], labels=CATEGORIES))
low, high = np.nanpercentile(boot, [2.5, 97.5])
print(f"95% CI: [{low:.3f}, {high:.3f}]")

# 4. Confusion matrix: rows = coder 1, columns = coder 2
confusion = pd.crosstab(c1, c2, rownames=["CD1"], colnames=["CD2"], dropna=False)
confusion = confusion.reindex(index=CATEGORIES, columns=CATEGORIES, fill_value=0)
print(confusion)

# 5. Per-category agreement (positive specific agreement)
rows = []
for cat in CATEGORIES:
    both = ((c1 == cat) & (c2 == cat)).sum()
    either = (c1 == cat).sum() + (c2 == cat).sum()
    rows.append({
        "category": cat,
        "coder1_n": (c1 == cat).sum(),
        "coder2_n": (c2 == cat).sum(),
        "agreement_%": round(2 * both / either * 100, 2) if either else np.nan,
    })
print(pd.DataFrame(rows))

Records: 3332
Percent agreement: 96.01%
Cohen's kappa: 0.931
95% CI: [0.920, 0.942]
CD2                            Compliance and responsibility  \
CD1                                                            
Compliance and responsibility                           1183   
Non applicability claims                                   0   
false positives                                           13   
Compliance artifacts                                      13   
Knowledge resources                                       17   
Reported alleged violations                                0   

CD2                            Non applicability claims  false positives  \
CD1                                                                        
Compliance and responsibility                         0                3   
Non applicability claims                              8                0   
false positives                                       1             1737   
Compliance artifacts   